# 🕵️ Proxy Design Pattern — Hinglish mein samjho (Beginner Friendly)

Is notebook mein hum **Proxy Design Pattern** ko step-by-step, Hinglish comments ke saath samjhenge.
Har code cell ke andar ek **multiline comment (docstring)** hai jisme concept Hinglish mein explain kiya gaya hai.

Pehle dekhenge **bina pattern ke** kya problem aati hai, phir **Proxy Pattern** se use solve karenge.
Aakhir mein ek **Practice Question** aur uska **Coding Solution** bhi diya gaya hai taaki tum khud practice kar sako.

In [1]:
"""
PROXY DESIGN PATTERN — KYA HAI?
=====================================

Proxy ek STRUCTURAL design pattern hai jisme ek "PROXY" object
ek PLACEHOLDER ki tarah kaam karta hai, aur REAL OBJECT ke
ACCESS ko CONTROL karta hai. Client SEEDHA Real Object se baat
NAHI karta -- wo PROXY se baat karta hai, aur PROXY request ko
AAGE Real Object ko FORWARD karta hai.

SOCHO EK EXAMPLE:
--------------------
Ek Image Viewer app mein HIGH-RESOLUTION images LOAD karna
EXPENSIVE hai. Iske bajaye ki APP TURANT image LOAD kare, ek
"ProxyImage" object use hota hai -- ye SIRF EK PLACEHOLDER hai.
Jab user ACTUALLY image OPEN kare (display() call ho), TABHI
Proxy REAL image ko LOAD karta hai. Agar user WAPAS dekhe, to
Proxy PEHLE SE LOADED image REUSE kar deta hai (CACHING).

MAIN COMPONENTS:
--------------------
1. Subject     -> COMMON interface jise RealSubject aur Proxy
                 DONO IMPLEMENT karte hain (e.g. Image)
2. RealSubject   -> ACTUAL object jo REAL kaam karta hai (e.g.
                 RealImage) -- ye EXPENSIVE ya SENSITIVE ho
                 sakta hai
3. Proxy           -> RealSubject ka PLACEHOLDER/SURROGATE.
                 Subject interface IMPLEMENT karta hai, RealSubject
                 ka REFERENCE rakhta hai, aur ACCESS CONTROL karta
                 hai (lazy loading, security check, caching, etc.)

KAISE KAAM KARTA HAI:
--------------------------
1. Client, Proxy se INTERACT karta hai (Real Object se SEEDHA NAHI)
2. Proxy ACCESS CONTROL karta hai (checks lagata hai)
3. Real Object SIRF TAB banaya/access kiya jaata hai jab ZAROORAT ho
4. Proxy REQUEST ko Real Object ko FORWARD karta hai

CHAINING OF PROXIES:
-------------------------
Proxies ko EK KE BAAD EK CHAIN mein bhi laga sakte hain -- HAR
proxy apna KAAM (check) karta hai, PHIR request ko AGLE proxy ya
FINALLY Real Object ko FORWARD karta hai. Jaise SECURITY GUARDS
ki EK CHAIN, HAR EK apna KAAM check karke AAGE bhejta hai.

REAL LIFE EXAMPLES:
-----------------------
- Virtual Proxy: Images/videos ko LAZILY load karna
- Remote Proxy: Kisi DOOSRE MACHINE par chal rahe object ka LOCAL
  representative (network calls ko HIDE karta hai)
- Security/Protection Proxy: USER PERMISSIONS check karke ACCESS
  dena (authentication/authorization systems)
- Caching Proxy: EXPENSIVE operations (DB query, API call) ke
  RESULTS CACHE karke REPEAT requests FAST karna

FAYDE (ADVANTAGES):
------------------------
- LAZY INITIALIZATION aur CACHING se PERFORMANCE IMPROVE hoti hai
- SENSITIVE objects ka ACCESS CONTROL karke SECURITY badhti hai
- Object creation DELAY karke RESOURCES SAVE hote hain

NUKSAAN (DISADVANTAGES):
----------------------------
- EXTRA classes se CODE COMPLEXITY badh jaati hai
- EXTRA INDIRECTION se THODA PERFORMANCE OVERHEAD ho sakta hai
- MULTIPLE LAYERS ki wajah se DEBUGGING MUSHKIL ho sakti hai
"""
print("Concept samajh liya! Ab neeche code mein implement karke dekhte hain.")


Concept samajh liya! Ab neeche code mein implement karke dekhte hain.


## ❌ Pehle dekho: BINA Proxy Pattern ke (Problem Code)

Chalo pehle ye dekhte hain ki Proxy Pattern KE BINA log Images ko kaise handle karte hain, aur
usme kya dikkat aati hai.

In [2]:
"""
BINA PROXY KE — PROBLEM DEMO
------------------------------------
Yahan "RealImageBad" object BANTE HI DISK se image LOAD kar leta
hai (simulate kiya hai print se -- real app mein ye SLOW file/
network read hoti). Agar HUMEIN SIRF EK IMAGE OBJECT BANANA ho
(display karna ho ya NA ho), TO BHI LOADING HO JAATI HAI.
"""

class RealImageBad:
    def __init__(self, filename):
        self.filename = filename
        self.load_image_from_disk()   # EXPENSIVE operation, __init__ mein hi ho gayi!

    def load_image_from_disk(self):
        print(f"⏳ Loading image '{self.filename}' from disk (EXPENSIVE)...")

    def display(self):
        print(f"🖼️ Displaying image: {self.filename}")


# Ek gallery mein 5 images ke NAAM hain, lekin user ne SIRF 1 image OPEN ki hai
image_filenames = ["photo1.jpg", "photo2.jpg", "photo3.jpg", "photo4.jpg", "photo5.jpg"]

print("Gallery load ho rahi hai (list bana rahe hain)...")
gallery = [RealImageBad(name) for name in image_filenames]   # SAARI 5 images LOAD ho gayin!

print("\nUser sirf 'photo1.jpg' dekhna chahta hai:")
gallery[0].display()


Gallery load ho rahi hai (list bana rahe hain)...
⏳ Loading image 'photo1.jpg' from disk (EXPENSIVE)...
⏳ Loading image 'photo2.jpg' from disk (EXPENSIVE)...
⏳ Loading image 'photo3.jpg' from disk (EXPENSIVE)...
⏳ Loading image 'photo4.jpg' from disk (EXPENSIVE)...
⏳ Loading image 'photo5.jpg' from disk (EXPENSIVE)...

User sirf 'photo1.jpg' dekhna chahta hai:
🖼️ Displaying image: photo1.jpg


In [3]:
"""
ISME PROBLEM KYA THI?
-------------------------
1. UNNECESSARY EAGER LOADING: Humne SIRF Gallery ki LIST BANAI
   thi (image OBJECTS banaye), lekin SAARI 5 images DISK SE
   LOAD ho gayin -- CHAHE user ne SIRF EK image DEKHI. Agar
   Gallery mein 1000 images hon, to app SHURU hote hi bahut
   SLOW ho jaayegi.

2. MEMORY WASTE: Jo images user NE KABHI DEKHI HI NAHI, WO BHI
   MEMORY mein LOADED baithi rahengi -- RESOURCE WASTE.

3. NO ACCESS CONTROL: Koi bhi code SEEDHA "RealImageBad(...)"
   bana sakta hai aur "display()" call kar sakta hai -- agar
   humein KABHI ACCESS CONTROL ya LOGGING ADD karni ho (jaise
   "sirf VERIFIED users hi image dekh sakein"), to HAR JAGAH,
   HAR CALLER mein ye LOGIC ADD karni padegi.

4. NO CACHING/REUSE MECHANISM: Agar HUM SAME image DUBARA
   "display()" karein, to (is example mein already loaded hai
   lekin) EK BADI file ke case mein AGAR RE-LOAD LOGIC ho, to
   HAR BAAR RE-LOAD hoti rehti, kyunki KOI CENTRAL PLACE NAHI
   hai jo "already loaded hai kya?" CHECK kare.

YE SAB PROBLEMS "Proxy Design Pattern" SOLVE KARTA HAI!
Neeche BEST APPROACH dekhte hain -- ek ProxyImage banayenge jo
REAL image ko SIRF TAB LOAD karega jab ACTUALLY zaroorat ho.
"""
print("Chalo ab BEST APPROACH (Proxy Pattern) dekhte hain...")


Chalo ab BEST APPROACH (Proxy Pattern) dekhte hain...


## ✅ Ab dekho: BEST APPROACH — Proxy Pattern se solve karte hain

### 1️⃣ Subject — `Image`

In [4]:
"""
SUBJECT (Image)
--------------------
Ye COMMON INTERFACE hai jise RealImage aur ProxyImage DONO
follow karenge. Isme SIRF EK abstract method hai: display().
"""

from abc import ABC, abstractmethod

class Image(ABC):
    @abstractmethod
    def display(self):
        pass


### 2️⃣ RealSubject — `RealImage`

In [5]:
"""
REAL SUBJECT (RealImage)
------------------------------
Ye ACTUAL, EXPENSIVE object hai. Iska constructor "disk se image
LOAD" karta hai -- ye SLOW operation hai. display() method image
ko DIKHATA hai.
"""

class RealImage(Image):
    def __init__(self, filename):
        self.filename = filename
        self.load_image_from_disk()

    def load_image_from_disk(self):
        print(f"⏳ Loading image '{self.filename}' from disk...")

    def display(self):
        print(f"🖼️ Displaying image: {self.filename}")


### 3️⃣ Proxy — `ProxyImage`

In [6]:
"""
PROXY (ProxyImage)
------------------------
Ye bhi Image interface IMPLEMENT karta hai -- isliye Client isko
"Image" hi samjhega. Lekin ANDAR SE ye RealImage ko SIRF TAB
BANATA hai jab "display()" PEHLI BAAR call ho (LAZY LOADING).
Agar DUBARA "display()" ho, to CACHED "real_image" ko REUSE
karta hai -- RE-LOAD NAHI hoti.
"""

class ProxyImage(Image):
    def __init__(self, filename):
        self.filename = filename
        self.real_image = None   # Abhi tak REAL image BANI HI NAHI

    def display(self):
        if self.real_image is None:
            print("♻️  Pehli baar display() call hua -- ab REAL image banegi")
            self.real_image = RealImage(self.filename)
        else:
            print("✅ Real image pehle se LOADED hai -- REUSE ho raha hai")
        self.real_image.display()


### 4️⃣ Client — Sirf Proxy se INTERACT karta hai

In [7]:
"""
CLIENT CODE
---------------
Ab Gallery ki list "ProxyImage" objects se BANAYENGE, RealImage
se NAHI. Object BANATE waqt DISK se KUCH LOAD NAHI hota -- SIRF
JAB "display()" call ho, TABHI REAL LOADING hogi.
"""

image_filenames = ["photo1.jpg", "photo2.jpg", "photo3.jpg", "photo4.jpg", "photo5.jpg"]

print("Gallery load ho rahi hai (SIRF proxies bana rahe hain, koi disk read NAHI)...")
gallery = [ProxyImage(name) for name in image_filenames]
print("(Dekha? Koi bhi 'Loading...' print NAHI hua -- kyunki RealImage abhi tak bana hi nahi!)\n")

print("User 'photo1.jpg' OPEN karta hai:")
gallery[0].display()

print("\nUser WAPAS 'photo1.jpg' dekhta hai:")
gallery[0].display()   # ab REUSE hoga, RE-LOAD nahi

print("\n✅ Proxy Pattern se sirf 1 image ACTUALLY load hui (jo user ne dekhi), baaki 4 KABHI load NAHI hui!")


Gallery load ho rahi hai (SIRF proxies bana rahe hain, koi disk read NAHI)...
(Dekha? Koi bhi 'Loading...' print NAHI hua -- kyunki RealImage abhi tak bana hi nahi!)

User 'photo1.jpg' OPEN karta hai:
♻️  Pehli baar display() call hua -- ab REAL image banegi
⏳ Loading image 'photo1.jpg' from disk...
🖼️ Displaying image: photo1.jpg

User WAPAS 'photo1.jpg' dekhta hai:
✅ Real image pehle se LOADED hai -- REUSE ho raha hai
🖼️ Displaying image: photo1.jpg

✅ Proxy Pattern se sirf 1 image ACTUALLY load hui (jo user ne dekhi), baaki 4 KABHI load NAHI hui!


## 📝 Practice Question (Tumhari baari!)

**Problem Statement:**

Ek **Bank Account Access Control System** banao using Proxy Design Pattern (jaise document ke
"Security Proxy for Access Control" real-life example mein bataya gaya tha).

`RealBankAccount` par sirf `"owner"` role hi withdraw kar sakta hai — koi aur role try kare to
`"Access Denied"` dikhna chahiye.

Tumhe ye banana hai (Image/RealImage/ProxyImage example jaisi hi structure use karo):

1. `BankAccount` (Subject) — abstract method `withdraw(amount)`
2. `RealBankAccount` (RealSubject) — `balance` attribute (start = 10000); `withdraw(amount)` mein agar `amount <= balance` to balance se minus karke print kare `"₹{amount} withdrawn. Balance: ₹{balance}"`, warna `"Insufficient funds"` print kare
3. `BankAccountProxy` (Proxy) — `user_role` aur `RealBankAccount` ka reference rakhe; `withdraw(amount)` mein CHECK kare ki `user_role == "owner"` — agar HAAN to `RealBankAccount.withdraw()` ko FORWARD kare, WARNA `"⛔ Access Denied for role: {user_role}"` print kare
4. Client code likho jo:
   - Ek `"guest"` role wala proxy banaye aur withdraw try kare (DENIED hona chahiye)
   - Ek `"owner"` role wala proxy banaye aur withdraw kare (SUCCESS hona chahiye)

👉 Pehle khud try karo neeche wale khaali code cell mein, phir uske baad wale cell mein solution dekho!

In [8]:
"""
YAHAN KHUD LIKHO (Practice Cell)
-----------------------------------
Hint: Image/RealImage/ProxyImage example ko copy karke sirf naam
badal do:
Image -> BankAccount, RealImage -> RealBankAccount, ProxyImage -> BankAccountProxy
display() -> withdraw(amount)

Apna code neeche likho:
"""

# TODO: class BankAccount (ABC) banao with abstract withdraw(amount)
# TODO: class RealBankAccount banao with balance, withdraw(amount)
# TODO: class BankAccountProxy banao with user_role check, withdraw(amount)
# TODO: Client code likho: guest se try karo (denied), owner se try karo (success)


'\nYAHAN KHUD LIKHO (Practice Cell)\n-----------------------------------\nHint: Image/RealImage/ProxyImage example ko copy karke sirf naam\nbadal do:\nImage -> BankAccount, RealImage -> RealBankAccount, ProxyImage -> BankAccountProxy\ndisplay() -> withdraw(amount)\n\nApna code neeche likho:\n'

## ✅ Solution — Practice Question ka Coding Solution

In [9]:
"""
SOLUTION EXPLANATION
-----------------------
Yahan bilkul Image/RealImage/ProxyImage jaisi hi structure use
ki hai, bas ab "Bank Account" bana rahe hain instead of "Image",
aur FOCUS "lazy loading" ki jagah "ACCESS CONTROL" hai.

1. BankAccount       -> Subject
2. RealBankAccount      -> RealSubject
3. BankAccountProxy       -> Proxy (yahan SECURITY CHECK karta hai)
4. Neeche client code hai jo GUEST aur OWNER dono test karta hai
"""

from abc import ABC, abstractmethod

# ---- 1. Subject ----
class BankAccount(ABC):
    @abstractmethod
    def withdraw(self, amount):
        pass


# ---- 2. Real Subject ----
class RealBankAccount(BankAccount):
    def __init__(self):
        self.balance = 10000

    def withdraw(self, amount):
        if amount <= self.balance:
            self.balance -= amount
            print(f"💸 ₹{amount} withdrawn. Balance: ₹{self.balance}")
        else:
            print("❌ Insufficient funds")


# ---- 3. Proxy (Access Control) ----
class BankAccountProxy(BankAccount):
    def __init__(self, user_role):
        self.user_role = user_role
        self.real_account = RealBankAccount()

    def withdraw(self, amount):
        if self.user_role != "owner":
            print(f"⛔ Access Denied for role: {self.user_role}")
            return
        print(f"✅ Access Granted for role: {self.user_role}")
        self.real_account.withdraw(amount)


# ---- 4. Client Code ----
print("Guest role withdraw karne ki koshish karta hai:")
guest_proxy = BankAccountProxy("guest")
guest_proxy.withdraw(2000)

print("\nOwner role withdraw karta hai:")
owner_proxy = BankAccountProxy("owner")
owner_proxy.withdraw(2000)


Guest role withdraw karne ki koshish karta hai:
⛔ Access Denied for role: guest

Owner role withdraw karta hai:
✅ Access Granted for role: owner
💸 ₹2000 withdrawn. Balance: ₹8000


## 🎯 Bonus Challenge

Ab try karo:
1. `BankAccountProxy` mein LOGGING add karo — HAR withdraw ATTEMPT (chahe SUCCESS ho ya DENIED)
   ko ek `transaction_log` list mein STORE karo, aur ek `show_log()` method banao.
2. Ek **CHAINED PROXY** try karo (document ke "Chaining of Proxies" concept se) — pehle ek
   `LoggingProxy` banao jo HAR call ko LOG kare, aur PHIR use `BankAccountProxy` ke SAATH
   CHAIN karo, taaki DONO checks (logging + access control) EK KE BAAD EK chalein.
3. Socho: agar `BankAccountProxy` NA hoti, aur `RealBankAccount` DIRECTLY expose hoti, to
   KOI BHI code role-check KIYE BINA seedha `withdraw()` call kar sakta tha. Ye kis
   REAL-WORLD security risk jaisa hai?